# PoliMillionaire — Kaggle live-play

Linear flow: **bootstrap → retrieval → LLM → play → persist**.
Set `COMPETITIONS` in the play cell to pick one or all four. The two
*Optional dev tools* cells in the middle are only used when iterating
on prompts; skip them for a normal run.

## 1. Bootstrap

Credentials, repo clone/pull, dependency install, indices symlink,
question-log pull from the Kaggle Dataset.

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# --- Secrets ---------------------------------------------------------------
secrets = UserSecretsClient()
gh_token = secrets.get_secret("GH_TOKEN")
os.environ["POLIMILLIONAIRE_API_URL"] = secrets.get_secret("POLIMILLIONAIRE_API_URL")
os.environ["POLIMILLIONAIRE_USER"] = secrets.get_secret("POLIMILLIONAIRE_USER")
os.environ["POLIMILLIONAIRE_PASSWORD"] = secrets.get_secret("POLIMILLIONAIRE_PASSWORD")
login(token=secrets.get_secret("HF_TOKEN"), add_to_git_credential=False)

# --- Repo: clone or fast-forward main --------------------------------------
REPO_DIR = Path("/kaggle/working/polimillionaire")
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    !git checkout -q main && git pull --ff-only origin main
else:
    !git clone -q https://{gh_token}@github.com/leonfuss/polimillionaire.git {REPO_DIR}
    os.chdir(REPO_DIR)

# --- Deps: install only what Kaggle's base image is missing ----------------
# Cached after first run, so warm kernels finish in <5 s.
NEEDED = {
    "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu",
    "bm25s": "bm25s",
    "polars": "polars",
    "dotenv": "python-dotenv",
    "sympy": "sympy",
}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    !pip install -q {" ".join(missing)}

# llama-cpp CUDA wheel is the one thing the UI can't pre-install for us.
if importlib.util.find_spec("llama_cpp") is None:
    !pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

# --- Project install (editable, no deps) -----------------------------------
!pip install -q -e {REPO_DIR} --no-deps
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

# --- Indices: symlink the uploaded FAISS/BM25 dataset into data/index ------
# Attach via Kaggle UI: Add data → search for your indices dataset.
# Kaggle mounts datasets at /kaggle/input/<slug>/ on most setups but at
# /kaggle/input/datasets/<owner>/<slug>/ for some shared-dataset configs.
# Try both so this works regardless.
INDEX_CANDIDATES = [
    Path("/kaggle/input/polimillionaire-indices"),
    Path("/kaggle/input/datasets/leonfuss/polimillionaire-indices"),
]
INDEX_SRC = next((p for p in INDEX_CANDIDATES if p.exists()), None)
INDEX_DST = REPO_DIR / "data" / "index"
INDEX_DST.parent.mkdir(parents=True, exist_ok=True)
if INDEX_SRC is not None and not INDEX_DST.exists():
    INDEX_DST.symlink_to(INDEX_SRC)

# --- Question log: pull latest version from the Kaggle Dataset -------------
# One-time setup: run scripts/kaggle_db_init.py locally to create the dataset.
from polimillionaire.kaggle_db import pull_db  # noqa: E402

# noqa: E402
db_path = pull_db(target_path="/kaggle/working/questions.sqlite")
os.environ["POLIMILLIONAIRE_DB_PATH"] = str(db_path)

# --- Sanity ----------------------------------------------------------------
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("indices:", sorted(p.name for p in (INDEX_DST.iterdir() if INDEX_DST.exists() else [])))
print("DB at:  ", db_path)

## 2. Retrieval models on `cuda:1`

Pin embedders + reranker to the second T4 so they don't fight
llama.cpp on `cuda:0`. ~10× faster reranking than CPU and zero
contention with the LLM.

In [ ]:
from polimillionaire.retrieval.embedder import Embedder
from polimillionaire.retrieval.reranker import DEFAULT_RERANKER, Reranker
from polimillionaire.strategies import factory

RETRIEVAL_DEVICE = "cuda:1"
for model in ("BAAI/bge-base-en-v1.5", "BAAI/bge-small-en-v1.5"):
    factory._embedder_cache[model] = Embedder(model, device=RETRIEVAL_DEVICE)
factory._reranker_cache[DEFAULT_RERANKER] = Reranker(DEFAULT_RERANKER, device=RETRIEVAL_DEVICE)
print("retrieval on", RETRIEVAL_DEVICE)

## 3. Load LLM

Qwen3-14B on `cuda:0`. First run downloads ~9 GB from HF; subsequent
runs hit the kernel-local cache.

In [ ]:
from polimillionaire.llm import load_llm

llm = load_llm("qwen3-14b")

## (Optional) Dev tools

Skip these for a normal run. Use only when iterating on prompts and
you want to avoid a full kernel restart after `git pull`.

In [ ]:
# Hot-reload: drop polimillionaire submodules from sys.modules so the
# next import re-reads from disk. Use after `git pull` to pick up
# prompt/code edits without restarting the kernel.
import sys

for mod_name in list(sys.modules):
    if mod_name.startswith("polimillionaire"):
        del sys.modules[mod_name]

print("index deleted. Make sure to reload the retrieval setup on cuda:1")

In [ ]:
# Render a sample prompt so you can eyeball what the model sees.
from polimillionaire._vendor.millionaire_client.models import Option, Question
from polimillionaire.prompts import calc_react, rag_calc_react

print("calc_react variants:    ", list(calc_react.PROMPTS))
print("rag_calc_react variants:", list(rag_calc_react.PROMPTS))
print()

q = Question(
    id=0,
    text="<placeholder>",
    options=[Option(id=i, text=f"opt {i}") for i in range(1, 5)],
    level=1,
)

# References=[] hides the retrieved-passages block. Pass real Passages
# from a Retriever.search(...) call to see what a live retrieval adds.
for m in rag_calc_react.PROMPTS["math-tir"].render(q, []):
    print(f"=== {m['role'].upper()} ===")
    print(m["content"])
    print()

## 4. Play

One configurable cell drives both **one competition** and **all four**.
Set `COMPETITIONS` to pick:

| Value | Plays |
| --- | --- |
| `[3]` | math only |
| `[0, 1, 2, 3]` | all four |
| `[2]` | science only (or `[0]` ent / `[1]` history) |

The math route uses `rag_calc_react` with the math-specialist
(`math-tir`) prompt; the other three use the `auto` router which
selects `wiki_rag` with the project's default prompt variant.

# 4a) Text Mode

In [ ]:
from polimillionaire import make_client, preload
from polimillionaire.play import auto_play_loop
from polimillionaire.strategies.factory import make_strategy

# --- config ----------------------------------------------------------------
COMPETITIONS = [0]  # [3] math only | [0, 1, 2, 3] all | [2] science only ...
MAX_GAMES = 12
MAX_STEPS = 3  # calc_react / rag_calc_react only
RAG_K = 3  # retrieved passages per math question
# ---------------------------------------------------------------------------

COMPETITION_NAMES = {0: "entertainment", 1: "history", 2: "science", 3: "math"}


def build_strategy(cid: int):
    """Per-competition strategy selection. Math gets the math-specialist
    prompt explicitly; the other three go through the auto router."""
    if cid == 3:
        return make_strategy(
            "rag_calc_react",
            llm,
            prompt_version="math-tir",
            verbose=True,
            max_steps=MAX_STEPS,
            k=RAG_K,
            db_retrieval=False,
            min_rerank_score=0.15,
        )
    return make_strategy(
        "wiki_rag",
        llm,
        competition_id=cid,
        verbose=True,
        max_steps=MAX_STEPS,
        min_rerank_score=0.15,
        db_retrieval=False,
    )


preload(COMPETITIONS)  # warm embedders + indices for the comps we'll play

results = {}
for cid in COMPETITIONS:
    name = COMPETITION_NAMES.get(cid, f"comp{cid}")
    print(f"\n========== {name} (competition {cid}) ==========")
    strategy = build_strategy(cid)
    results[name] = auto_play_loop(make_client(), cid, strategy, max_games=MAX_GAMES)

# --- summary ---------------------------------------------------------------
print("\n=== summary ===")
for name, s in results.items():
    total = s["correct"] + s["wrong"]
    pct = 100 * s["correct"] / total if total else 0
    print(f"  {name:14s}  {s['correct']}/{total} ({pct:.0f}%)  timeouts={s['timeouts']}")

## 4b) Speech Mode

In [ ]:
from polimillionaire import make_client, preload
from polimillionaire.asr import WhisperTranscriber
from polimillionaire.play import speech_auto_play_loop

# --- config ----------------------------------------------------------------
COMPETITIONS = [3]  # speech-mode works on any comp; pick what you want to play
MAX_GAMES = 20  # speech adds ~5-15s of audio fetch + ASR per question,
# so games run noticeably slower than text mode
MAX_STEPS = 3  # calc_react / rag_calc_react only
RAG_K = 3  # retrieved passages per math question
# ---------------------------------------------------------------------------

COMPETITION_NAMES = {0: "entertainment", 1: "history", 2: "science", 3: "math"}


def build_speech_strategy(cid: int):
    """Same per-competition routing as text mode, but with mode='speech'
    and use_text_mode_retrieval=True so the wrapper leans on the DB rows
    we built up during prior text-mode runs while speech-mode rows are
    still sparse."""
    common = dict(
        verbose=True,
        max_steps=MAX_STEPS,
        min_rerank_score=0.15,
        db_retrieval=True,
        mode="speech",
        use_text_mode_retrieval=True,
    )
    if cid == 3:
        return make_strategy(
            "rag_calc_react",
            llm,
            prompt_version="math-tir",
            k=RAG_K,
            **common,
        )
    return make_strategy("auto", llm, competition_id=cid, **common)


# Whisper-large-v3-turbo on cuda:1 (override with POLIMILLIONAIRE_ASR_DEVICE
# if your layout differs). Eager-load so the first question doesn't pay the
# ~1s HF download + model-load tax inside the 30s timer.
transcriber = WhisperTranscriber()
transcriber.preload()

preload(COMPETITIONS)  # warm embedders + indices for the comps we'll play

results = {}
for cid in COMPETITIONS:
    name = COMPETITION_NAMES.get(cid, f"comp{cid}")
    print(f"\n========== {name} (competition {cid}, speech) ==========")
    strategy = build_speech_strategy(cid)
    results[name] = speech_auto_play_loop(
        make_client(), cid, strategy, transcriber, max_games=MAX_GAMES
    )

# --- summary ---------------------------------------------------------------
print("\n=== summary ===")
for name, s in results.items():
    total = s["correct"] + s["wrong"]
    pct = 100 * s["correct"] / total if total else 0
    print(f"  {name:14s}  {s['correct']}/{total} ({pct:.0f}%)  timeouts={s['timeouts']}")

## 5. Persist the question log

Push the updated SQLite log back to the Kaggle Dataset as a new
version. The next kernel session will pull this version automatically.
**Run this before stopping the session** — `/kaggle/working/` is
wiped at session end.

In [ ]:
from polimillionaire.kaggle_db import push_db  # noqa: E402

# push_db now pulls the latest remote, merges any new local rows in (dedup
# by account_username + session_id + level + mode), then uploads the union.
# Multiple kernels can play concurrently and converge -- see the docstring
# for the consistency guarantees.
stats = push_db("/kaggle/working/questions.sqlite", version_notes="kaggle live-play")
print(
    f"DB pushed: +{stats['inserted']} new rows merged from this kernel, "
    f"{stats['skipped']} already in remote."
)